Processing of data in ATSBS. Note the any modification of the file without the use of this notebook requires cell [3] to be rerun for this notebook to properly.

Open "Average_Transit_Speeds_by_Stop_Segments.csv", make can clone, and save it as "atsbs.csv"

In [75]:
import pandas as pd

# Read the original CSV file
df = pd.read_csv("Average_Transit_Speeds_by_Stop_Segments.csv")

# Save a clone as "atsbs.csv"
df.to_csv("atsbs.csv", index=False)

Open "atsbs.csv" with Pandas

In [76]:
import pandas as pd

df = pd.read_csv("atsbs.csv")

Drop all unneeded fields: {"route_id", "stop_pair", "base64_url", "org_id", "agency", "Shape_Length"}

In [77]:
df = df.drop(columns=["route_id", "stop_pair", "base64_url", "org_id", "agency", "Shape_Length"])
df.to_csv("atsbs.csv", index=False)

Reformat the "district_name" so that it only contains the number eg. "NN" instead of "NN - XXXXXX"

In [78]:
df["district_name"] = df["district_name"].str[:2]
df.to_csv("atsbs.csv", index=False)

Sort the data by "district_name" then "stop_pair_name" then "OBJECTID" then "direction_id" then "time_period"

In [ ]:
df = df.sort_values(by=["district_name", "stop_pair_name", "OBJECTID", "direction_id", "time_period"])
df.to_csv("atsbs.csv", index=False)

Remove the data if the "district_name" is empty

In [80]:
df = df[df["district_name"].str.strip() != ""]
df.to_csv("atsbs.csv", index=False)

Sort the columns by "OBJECTID" then "stop_pair_name" then "direction_id" then "time_period" then "p50_mph" then "n_trips" then "district_name" then "p20_mph" then "p80_mph"

In [81]:
df = df[["OBJECTID", "stop_pair_name", "direction_id", "time_period", "p50_mph", "n_trips", "district_name", "p20_mph", "p80_mph"]]
df.to_csv("atsbs.csv", index=False)

Remove the data if the "time_period" is "all_day"

In [82]:
df = df[df["time_period"] != "all_day"]
df.to_csv("atsbs.csv", index=False)

Remove standalone data. Meaning that if the OBJECTID of the row above is not the (OBJECTID of the current - 1) and the OBJECTID of the row below is not the (OBJECTID of the current + 1)

In [83]:
# Identify rows where the OBJECTID of the previous row is (current OBJECTID - 1)
prev_is_consecutive = df["OBJECTID"].shift(1) == (df["OBJECTID"] - 1)
# Identify rows where the OBJECTID of the next row is (current OBJECTID + 1)
next_is_consecutive = df["OBJECTID"].shift(-1) == (df["OBJECTID"] + 1)
# Keep rows that are not standalone
df = df[prev_is_consecutive | next_is_consecutive]
df.to_csv("atsbs.csv", index=False)

Pick the percentile to calculate: "p20_mph", "p50_mph", "p80_mph"

In [84]:
p2use = "p80_mph"
p2cr = "p80_cr"
p2_acr = "p80_acr"
percentiles = ["p20_mph", "p50_mph", "p80_mph"]
percentiles.remove(p2use)

Drop the unneeded speed percentile columns and keep pX_mph. (eg Drop "p20_mph", "p80_mph" and keep "p50_mph")

In [85]:
df = df.drop(columns=percentiles)
df.to_csv("atsbs.csv", index=False)

Drop the "OBJECTID" and "time_period" column

In [86]:
df = df.drop(columns=["OBJECTID", "time_period"])
df.to_csv("atsbs.csv", index=False)

Merge the data to calculate congestion rate by formulae: [(pX_mph_offpeak) - (pX_mph_peak) / (pX_mph_offpeak)];[(total n_trips) = (n_trips top) + (n_trips bottom)]

In [87]:
# Create a new DataFrame by grouping every two consecutive rows
merged_rows = []

for i in range(0, len(df) - 1, 2):
    top = df.iloc[i]
    bottom = df.iloc[i + 1]
    merged = {
        "stop_pair_name": top["stop_pair_name"],
        "direction_id": top["direction_id"],
        "district_name": top["district_name"],
        "n_trips": top["n_trips"] + bottom["n_trips"],
        p2use: (top[p2use] - bottom[p2use]) / top[p2use] if top[p2use] != 0 else None
    }
    merged_rows.append(merged)

merged_df = pd.DataFrame(merged_rows)
merged_df.to_csv("atsbs.csv", index=False)

Drop columns {"stop_pair_name", "direction_id"}

In [88]:
merged_df = merged_df.drop(columns=["stop_pair_name", "direction_id"])
merged_df.to_csv("atsbs.csv", index=False)

Change field name of "pX_mph" to "pX_cr", pX congestion rate.

In [89]:
merged_df = merged_df.rename(columns={p2use: p2cr})
merged_df.to_csv("atsbs.csv", index=False)

Merge the rows to find aggregate congestion rate for each "district_name" based on the formula: (District Congestion Index)=[∑(n_trips(i))*​(Segment Congestion Index(i))/∑(n_trips(i))]

In [90]:
# Group by 'district_name' and calculate the weighted average congestion index
district_agg = merged_df.groupby("district_name").apply(
    lambda g: pd.Series({
        "district_n_trips": g["n_trips"].sum(),
        "district_congestion_index": (g["n_trips"] * g[p2cr]).sum() / g["n_trips"].sum()
    })
).reset_index()

district_agg.to_csv("atsbs.csv", index=False)

/var/folders/0g/g1btf_jx2p1f1dx9sx5n8c380000gp/T/ipykernel_15582/3384667736.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  district_agg = merged_df.groupby("district_name").apply(


Drop the "district_n_trips" column

In [91]:
district_agg = district_agg.drop(columns=["district_n_trips"])
district_agg.to_csv("atsbs.csv", index=False)

Change field name of "district_congestion_index" to "pX_acr", pX aggregate congestion rate.

In [92]:
district_agg = district_agg.rename(columns={"district_congestion_index": p2_acr})
district_agg.to_csv("atsbs.csv", index=False)